# 02 — Test vol-engine (Redis outputs : surface SVI/SSVI + signaux + heartbeat)

Smoke test du container `fxvol-vol-engine` — étape 2/5. Valide que **les sorties Redis du cycle 180s sont bien produites** : heartbeat ISO, `latest_vol_surface:EURUSD` JSON avec la structure attendue (pillars δ + RV/HAR/GARCH + SVI par tenor + SSVI surface + `_fair_q`), et `latest_signals:EURUSD` avec une liste non-vide de signaux CHEAP/FAIR/EXPENSIVE.

**Setup particulier** : contrairement à risk-engine (cycle 2s, instantané), vol-engine cycle = **180s** + 5-10s de calc (GARCH/SVI/SSVI). Le notebook 01 a déjà attendu start_period 240s pour le 1er cycle, donc à ce stade on **devrait** avoir des outputs en place. Ce notebook vérifie d'abord si la surface est fraîche, sinon attend jusqu'à 200s qu'un nouveau cycle complète.

**Pré-requis IB Gateway healthy** : le cycle vol skip si `_fetch_fop_chain` renvoie vide (IB déconnecté, hors heures de marché, TrustedIPs KO). Si ce notebook FAIL en §2 alors que 01 est OK, c'est probablement IB Gateway qui ne sert pas la chain — voir notebook ib-gateway/02 ou 04.

**Note hors-marché FX** : EURUSD est tradé H24 du dimanche 22h UTC au vendredi 22h UTC. En weekend (samedi + dimanche avant 22h) la chain peut être vide même Gateway healthy. Dans ce cas, dégrader le test en §2 = WARN plutôt que FAIL.

**Couvre** :
1. Seed `latest_spot:EURUSD` si market-data ne le pousse pas (sinon skip)
2. `heartbeat:vol_engine` frais (ISO-8601, age < 200s, TTL > 0)
3. `latest_vol_surface:EURUSD` JSON valide avec `{symbol, timestamp, surface}` au top-level
4. `surface` contient ≥ 1 tenor avec ≥ 1 pillar (idéalement les 5 : 10dp/25dp/atm/25dc/10dc)
5. `_svi` per-tenor avec params plausibles (`a, b, rho ∈ [-1,1], sigma > 0, rmse_fit, butterfly_g_min`)
6. `_har`/`_garch`/`_rv_full_pct` présents (P-measure estimators) — soft check, peut manquer si OHLC fetch fail
7. `_fair_q` per-tenor avec `sigma_fair_p_pct, vrp_vol_pts, sigma_fair_q_pct, regime`
8. `_ssvi` (soft) — surface fit, peut être absent si < 2 tenors avec ≥ 5 obs
9. `latest_signals:EURUSD` JSON valide avec liste non-vide de signaux
10. Chaque signal : `signal_type ∈ {CHEAP, FAIR, EXPENSIVE}`, `sigma_mid/sigma_fair` floats finis, `dte > 0`
11. Cohérence : un signal par tenor avec `atm` dans la surface

**Préreq** :
- Notebook 01 vert (container UP + healthy)
- ib-gateway healthy + TrustedIPs OK (172.20.0.11 dans la liste)
- Heures de marché FX (sinon dégrader les attentes — voir §2)
- Redis exposé sur `localhost:6380`

## Setup + seed spot si absent

Vol-engine lit `latest_spot:EURUSD` (publié par market-data). Si market-data est down, on seed un spot synthétique pour ne pas bloquer le test ; mais on note le warning car ça veut dire que le pipeline réel est cassé.

In [23]:
import json
import math
import time
from datetime import datetime, timezone

import redis

REDIS_URL = "redis://localhost:6380/0"
SYMBOL = "EURUSD"

# Cycle vol = 180s, +5-10s GARCH/SVI/SSVI. 200s = 1 cycle nominal complet.
MAX_WAIT_FOR_CYCLE_S = 200.0
POLL_S = 5.0

EXPECTED_PILLAR_LABELS = {"10dp", "25dp", "atm", "25dc", "10dc"}
EXPECTED_SIGNAL_TYPES = {"CHEAP", "FAIR", "EXPENSIVE"}

results = []

def record(label, ok, detail=""):
    results.append((label, ok, detail))
    sym = "OK" if ok else "FAIL"
    print(f"  [{sym:4}] {label}{('  | ' + detail) if detail else ''}")

r = redis.from_url(REDIS_URL, decode_responses=True)
if not r.ping():
    raise RuntimeError("Redis ping FAIL — check 'docker compose ps'")
print(f"Connected -> {REDIS_URL}")

# Spot pré-existant ?
spot_raw = r.get(f"latest_spot:{SYMBOL}")
if spot_raw is None:
    print("  [WARN] latest_spot:EURUSD absent — market-data en panne ?")
    print("         Seed synthétique 1.17000 (vol-engine accepte plain float)")
    r.set(f"latest_spot:{SYMBOL}", "1.17000", ex=600)
else:
    print(f"  [INFO] latest_spot:{SYMBOL} = {spot_raw[:60]}")

Connected -> redis://localhost:6380/0
  [INFO] latest_spot:EURUSD = 1.169915


## 1. Heartbeat frais — wait for one cycle si nécessaire

Si le heartbeat existe et a < 200s d'âge, on accepte tel quel. Sinon on poll jusqu'à 200s qu'un nouveau cycle pousse un heartbeat frais.

**Si le heartbeat ne se rafraîchit jamais** : engine en cycle skip permanent (spot manquant, surface vide, IB chain vide). Voir logs `docker logs fxvol-vol-engine --tail 100`.

In [24]:
print("== 1. heartbeat ==")

def hb_age():
    raw = r.get("heartbeat:vol_engine")
    if not raw:
        return None, None
    try:
        ts = datetime.fromisoformat(raw.replace("Z", "+00:00"))
        return raw, (datetime.now(timezone.utc) - ts).total_seconds()
    except ValueError:
        return raw, None

raw, age = hb_age()
if raw is None or age is None or age > MAX_WAIT_FOR_CYCLE_S:
    print(f"  [INFO] heartbeat absent ou stale (age={age}) — wait up to {MAX_WAIT_FOR_CYCLE_S}s pour un cycle frais")
    deadline = time.time() + MAX_WAIT_FOR_CYCLE_S
    while time.time() < deadline:
        time.sleep(POLL_S)
        raw, age = hb_age()
        if raw is not None and age is not None and age < MAX_WAIT_FOR_CYCLE_S:
            break

record("heartbeat:vol_engine présent",
       raw is not None,
       raw or "<missing — engine en cycle skip ? logs : docker logs fxvol-vol-engine --tail 100>")
if raw and age is not None:
    record("heartbeat ISO-8601 parsable", True, f"parsed = {raw}")
    record(f"heartbeat age < {MAX_WAIT_FOR_CYCLE_S}s",
           age < MAX_WAIT_FOR_CYCLE_S, f"age = {age:.1f}s")
    ttl = r.ttl("heartbeat:vol_engine")
    record("heartbeat TTL > 0", ttl > 0, f"TTL = {ttl}s")

== 1. heartbeat ==
  [INFO] heartbeat absent ou stale (age=None) — wait up to 200.0s pour un cycle frais
  [OK  ] heartbeat:vol_engine présent  | 2026-04-29T09:21:24.280240Z
  [OK  ] heartbeat ISO-8601 parsable  | parsed = 2026-04-29T09:21:24.280240Z
  [OK  ] heartbeat age < 200.0s  | age = 0.4s
  [OK  ] heartbeat TTL > 0  | TTL = 30s


## 2. `latest_vol_surface:EURUSD` — top-level schéma

**Schéma attendu** (cf. `bus/publisher.py:publish_vol_update`) :
```json
{
  "symbol": "EURUSD",
  "timestamp": "ISO-8601 UTC",
  "surface": {
    "1M": { "10dp": {iv, strike}, "25dp": ..., "atm": ..., "25dc": ..., "10dc": ... },
    "3M": { ... },
    "_rv_full_pct": 7.2,
    "_har": { "1M": {sigma_har_pct, ...}, ... },
    "_garch": { ... },
    "_fair_q": { "1M": {sigma_fair_p_pct, vrp_vol_pts, sigma_fair_q_pct, regime}, ... },
    "_svi": { "1M": {a, b, rho, m, sigma, rmse_fit, butterfly_g_min}, ... },
    "_ssvi": {eta, gamma, rho, rmse}
  }
}
```

In [25]:
print("== 2. latest_vol_surface top-level ==")

surf_raw = r.get(f"latest_vol_surface:{SYMBOL}")
record("latest_vol_surface:EURUSD présent",
       surf_raw is not None,
       (surf_raw[:80] + "...") if surf_raw and len(surf_raw) > 80
       else (surf_raw or "<missing — IB chain vide ? hors heures FX ? logs vol-engine>"))

surface_payload = None
if surf_raw:
    try:
        surface_payload = json.loads(surf_raw)
        record("latest_vol_surface JSON valide", True, f"keys = {sorted(surface_payload.keys())}")
    except json.JSONDecodeError as e:
        record("latest_vol_surface JSON valide", False, str(e))

if surface_payload:
    has_top = {"symbol", "timestamp", "surface"} <= set(surface_payload.keys())
    record("top-level keys = {symbol, timestamp, surface}", has_top,
           f"actual = {sorted(surface_payload.keys())}")
    record("top-level symbol = EURUSD",
           surface_payload.get("symbol") == "EURUSD",
           f"got {surface_payload.get('symbol')!r}")

surface = (surface_payload or {}).get("surface", {}) if surface_payload else {}

== 2. latest_vol_surface top-level ==
  [OK  ] latest_vol_surface:EURUSD présent  | {"symbol": "EURUSD", "timestamp": "2026-04-29T09:21:24.277633Z", "surface": {"1M...
  [OK  ] latest_vol_surface JSON valide  | keys = ['surface', 'symbol', 'timestamp']
  [OK  ] top-level keys = {symbol, timestamp, surface}  | actual = ['surface', 'symbol', 'timestamp']
  [OK  ] top-level symbol = EURUSD  | got 'EURUSD'


## 3. Surface : tenors + pillars

Tenors attendus : 1M / 2M / 3M / 4M / 5M / 6M (cf. `DEFAULT_TENOR_T` dans `engine.py`). Avec un compte paper IB et des FOP EURUSD, on a typiquement 4-6 tenors qui ressortent (les autres peuvent être vides faute de strikes listés).

Pour un tenor, les pillars (PCHIP-interpolés à partir des observations IB) sont 10dp / 25dp / atm / 25dc / 10dc. On accepte 3+ pillars présents par tenor (le SVI fit en exige ≥ 3).

In [26]:
print("== 3. tenors + pillars ==")

if not surface:
    record("surface non-vide", False, "skip (cf. §2)")
else:
    public_tenors = [t for t in surface if not t.startswith("_") and isinstance(surface[t], dict)]
    record("≥ 1 tenor public dans surface", len(public_tenors) >= 1,
           f"tenors = {public_tenors}")

    for tenor in public_tenors:
        pillars = surface[tenor]
        present_labels = {lab for lab in EXPECTED_PILLAR_LABELS if isinstance(pillars.get(lab), dict)}
        record(f"  tenor {tenor} a ≥ 3 pillars",
               len(present_labels) >= 3,
               f"pillars = {sorted(present_labels)}")
        # Sanity sur le pillar atm
        atm = pillars.get("atm") if isinstance(pillars.get("atm"), dict) else {}
        iv = atm.get("iv")
        strike = atm.get("strike")
        atm_ok = (
            isinstance(iv, (int, float)) and 0 < iv < 1
            and isinstance(strike, (int, float)) and 0.5 < strike < 2.0
        )
        record(f"  tenor {tenor}.atm cohérent (0<iv<1, 0.5<strike<2)",
               atm_ok, f"iv={iv}, strike={strike}")

== 3. tenors + pillars ==
  [OK  ] ≥ 1 tenor public dans surface  | tenors = ['1M', '2M', '3M', '4M', '5M', '6M']
  [OK  ]   tenor 1M a ≥ 3 pillars  | pillars = ['10dc', '10dp', '25dc', '25dp', 'atm']
  [OK  ]   tenor 1M.atm cohérent (0<iv<1, 0.5<strike<2)  | iv=0.05896932281195571, strike=1.1724348009963868
  [OK  ]   tenor 2M a ≥ 3 pillars  | pillars = ['10dc', '10dp', '25dc', '25dp', 'atm']
  [OK  ]   tenor 2M.atm cohérent (0<iv<1, 0.5<strike<2)  | iv=0.0593709569288223, strike=1.1764884947704999
  [OK  ]   tenor 3M a ≥ 3 pillars  | pillars = ['10dc', '10dp', '25dc', '25dp', 'atm']
  [OK  ]   tenor 3M.atm cohérent (0<iv<1, 0.5<strike<2)  | iv=0.05981060811530449, strike=1.1764779642437415
  [OK  ]   tenor 4M a ≥ 3 pillars  | pillars = ['10dc', '10dp', '25dc', '25dp', 'atm']
  [OK  ]   tenor 4M.atm cohérent (0<iv<1, 0.5<strike<2)  | iv=0.06001395687870392, strike=1.1765912099368556
  [OK  ]   tenor 5M a ≥ 3 pillars  | pillars = ['10dc', '10dp', '25dc', '25dp', 'atm']
  [OK  ]   tenor

## 4. SVI fits par tenor

Pour chaque tenor avec ≥ 3 pillars, on attend un fit SVI (raw : a, b, rho, m, sigma) + métriques (rmse_fit, butterfly_g_min).

**Bornes plausibles** :
- `a` : ~0 à quelques pourcents², peut être très petit
- `b` : > 0
- `rho` : ∈ [-1, 1] strict (≃ skew sign)
- `m` : log-moneyness centre, typiquement |m| < 0.5
- `sigma` : > 0, typiquement < 1
- `rmse_fit` : < 0.01 (en variance totale, donc petit)
- `butterfly_g_min` : ≥ 0 ⇒ pas de violation densité ; négatif = WARN (logs vol-engine `svi_butterfly_violation`)

In [27]:
print("== 4. SVI per-tenor ==")

svi_all = surface.get("_svi") or {}
record("_svi présent et non-vide", bool(svi_all), f"tenors = {sorted(svi_all)}")

for tenor, p in svi_all.items():
    has_keys = {"a", "b", "rho", "m", "sigma", "rmse_fit", "butterfly_g_min"} <= set(p.keys())
    record(f"  svi[{tenor}] keys complets", has_keys, f"got {sorted(p.keys())}")
    if not has_keys:
        continue
    rho_ok = isinstance(p["rho"], (int, float)) and -1 <= p["rho"] <= 1
    sigma_ok = isinstance(p["sigma"], (int, float)) and p["sigma"] > 0
    b_ok = isinstance(p["b"], (int, float)) and p["b"] > 0
    rmse_ok = isinstance(p["rmse_fit"], (int, float)) and 0 <= p["rmse_fit"] < 0.05
    record(f"  svi[{tenor}] rho ∈ [-1,1]", rho_ok, f"rho={p['rho']}")
    record(f"  svi[{tenor}] sigma > 0", sigma_ok, f"sigma={p['sigma']}")
    record(f"  svi[{tenor}] b > 0", b_ok, f"b={p['b']}")
    record(f"  svi[{tenor}] rmse_fit < 0.05", rmse_ok, f"rmse={p['rmse_fit']}")
    g_min = p["butterfly_g_min"]
    if isinstance(g_min, (int, float)) and g_min < 0:
        # Soft : flagué WARN, mais ne fait pas échouer le smoke. Le bug fix
        # se fait côté SVI fit, pas côté smoke.
        print(f"  [WARN] svi[{tenor}].butterfly_g_min = {g_min:.4f} < 0 (densité négative)")

== 4. SVI per-tenor ==
  [OK  ] _svi présent et non-vide  | tenors = ['1M', '2M', '3M', '4M', '5M', '6M']
  [OK  ]   svi[1M] keys complets  | got ['a', 'b', 'butterfly_g_min', 'm', 'rho', 'rmse_fit', 'sigma']
  [OK  ]   svi[1M] rho ∈ [-1,1]  | rho=-0.735605
  [OK  ]   svi[1M] sigma > 0  | sigma=0.031349
  [OK  ]   svi[1M] b > 0  | b=0.012046
  [OK  ]   svi[1M] rmse_fit < 0.05  | rmse=1.1e-05
  [OK  ]   svi[2M] keys complets  | got ['a', 'b', 'butterfly_g_min', 'm', 'rho', 'rmse_fit', 'sigma']
  [OK  ]   svi[2M] rho ∈ [-1,1]  | rho=-0.514148
  [OK  ]   svi[2M] sigma > 0  | sigma=0.043472
  [OK  ]   svi[2M] b > 0  | b=0.015912
  [OK  ]   svi[2M] rmse_fit < 0.05  | rmse=1.2e-05
  [OK  ]   svi[3M] keys complets  | got ['a', 'b', 'butterfly_g_min', 'm', 'rho', 'rmse_fit', 'sigma']
  [OK  ]   svi[3M] rho ∈ [-1,1]  | rho=-0.588073
  [OK  ]   svi[3M] sigma > 0  | sigma=0.050307
  [OK  ]   svi[3M] b > 0  | b=0.022298
  [OK  ]   svi[3M] rmse_fit < 0.05  | rmse=2.5e-05
  [OK  ]   svi[4M] keys com

## 5. Estimateurs P-measure : RV / HAR / GARCH (strict avec abonnement CME NLP L1)

Avec un abonnement IB **CME Real-Time NonProfessional Level 1** (le compte courant), l'historical EUR/USD futures **doit** être disponible — donc RV / HAR / GARCH doivent tous être présents et finis.

`_rv_full_pct` typiquement 4-12% pour EURUSD daily. `_har`/`_garch` chacun avec ≥ 1 tenor fitté.

Si ce test FAIL : bug dans `historical_fetcher.py` (mauvais `whatToShow`, mauvais contract type), ou perms IB historical réellement absentes (à vérifier en console IB).

In [28]:
print("== 5. RV / HAR / GARCH ==")

rv = surface.get("_rv_full_pct")
record("_rv_full_pct présent",
       isinstance(rv, (int, float)) and not math.isnan(rv),
       f"rv = {rv}" if rv is not None else "<missing — check historical_fetcher logs>")
if isinstance(rv, (int, float)):
    record("_rv_full_pct dans plage plausible (1-30%)",
           1.0 < rv < 30.0, f"rv = {rv}")

har = surface.get("_har") or {}
garch = surface.get("_garch") or {}
record("_har OU _garch non-vide",
       bool(har) or bool(garch),
       f"har_tenors={list(har)}, garch_tenors={list(garch)}")

== 5. RV / HAR / GARCH ==
  [FAIL] _rv_full_pct présent  | <missing — check historical_fetcher logs>
  [FAIL] _har OU _garch non-vide  | har_tenors=[], garch_tenors=[]


## 6. `_fair_q` per-tenor (conversion P → Q via VRP)

Pour chaque tenor avec un sigma_fair^P calculé (HAR ou GARCH), on attend un node `_fair_q[tenor] = {sigma_fair_p_pct, vrp_vol_pts, sigma_fair_q_pct, regime}`.

**Bornes** :
- `sigma_fair_p_pct` : > 0 (typiquement 3-15% pour FX vanilla)
- `vrp_vol_pts` : peut être négatif ou positif (régime-dependent)
- `sigma_fair_q_pct` = sigma_fair_p_pct + vrp_vol_pts, > 0
- `regime` ∈ {ex: "calm", "normal", "stress"} — string non-vide

Soft : si `_har`/`_garch` absents, `_fair_q` est attendu vide → on skip ce test. Sinon `_fair_q` ne doit pas être vide.

In [29]:
print("== 6. _fair_q per-tenor ==")

fair_q = surface.get("_fair_q") or {}
record("_fair_q non-vide",
       bool(fair_q),
       f"got {sorted(fair_q)}")

for tenor, fq in fair_q.items():
    keys_ok = {"sigma_fair_p_pct", "vrp_vol_pts", "sigma_fair_q_pct", "regime"} <= set(fq.keys())
    record(f"  fair_q[{tenor}] keys complets", keys_ok, f"got {sorted(fq.keys())}")
    if not keys_ok:
        continue
    p_pct = fq["sigma_fair_p_pct"]
    q_pct = fq["sigma_fair_q_pct"]
    vrp = fq["vrp_vol_pts"]
    delta = abs(q_pct - (p_pct + vrp))
    record(f"  fair_q[{tenor}] : q ≈ p + vrp", delta < 0.01,
           f"p={p_pct}, vrp={vrp}, q={q_pct}, delta={delta:.4f}")
    record(f"  fair_q[{tenor}] regime non-vide",
           isinstance(fq["regime"], str) and fq["regime"],
           f"regime={fq['regime']!r}")

== 6. _fair_q per-tenor ==
  [FAIL] _fair_q non-vide  | got []


## 7. SSVI surface-wide (soft)

Présent uniquement si ≥ 2 tenors avec atm + ≥ 5 observations agrégées. Bornes :
- `eta > 0`, `0 < gamma < 1`, `-1 < rho < 1`, `rmse < 0.05`

In [30]:
print("== 7. SSVI surface-wide ==")

ssvi = surface.get("_ssvi")
if not ssvi:
    print("  [SKIP] _ssvi absent — < 2 tenors atm ou < 5 obs (peut être OK selon chain IB)")
else:
    record("_ssvi présent", True, f"keys = {sorted(ssvi)}")
    eta = ssvi.get("eta")
    gamma = ssvi.get("gamma")
    rho = ssvi.get("rho")
    rmse = ssvi.get("rmse") or ssvi.get("rmse_fit")
    record("  ssvi.eta > 0", isinstance(eta, (int, float)) and eta > 0, f"eta={eta}")
    record("  ssvi.gamma ∈ (0, 1)",
           isinstance(gamma, (int, float)) and 0 < gamma < 1, f"gamma={gamma}")
    record("  ssvi.rho ∈ (-1, 1)",
           isinstance(rho, (int, float)) and -1 < rho < 1, f"rho={rho}")
    if rmse is not None:
        record("  ssvi.rmse < 0.05",
               isinstance(rmse, (int, float)) and 0 <= rmse < 0.05, f"rmse={rmse}")

== 7. SSVI surface-wide ==
  [OK  ] _ssvi présent  | keys = ['calendar_arb_free', 'eta', 'gamma', 'rho', 'rmse_fit']
  [OK  ]   ssvi.eta > 0  | eta=1.706729
  [OK  ]   ssvi.gamma ∈ (0, 1)  | gamma=0.407329
  [OK  ]   ssvi.rho ∈ (-1, 1)  | rho=-0.110978
  [OK  ]   ssvi.rmse < 0.05  | rmse=1.8e-05


## 8. `latest_signals:EURUSD` — schéma + signaux

**Schéma attendu** (cf. `_derive_signals`) :
```json
{
  "symbol": "EURUSD",
  "timestamp": "ISO-8601",
  "signals": [
    {"underlying":"EURUSD", "tenor":"1M", "dte":30, "sigma_mid":7.05, "sigma_fair":6.80,
     "ecart":0.25, "signal_type":"FAIR", "rv":7.2, "sigma_fair_p":7.1, "vrp_vol_pts":-0.30}
  ]
}
```
Un signal par tenor avec un pillar `atm` valide. Si la surface n'a pas de tenor public (cas extrême), la liste peut être vide.

In [31]:
print("== 8. latest_signals ==")

sig_raw = r.get(f"latest_signals:{SYMBOL}")
record("latest_signals:EURUSD présent",
       sig_raw is not None,
       (sig_raw[:80] + "...") if sig_raw and len(sig_raw) > 80 else (sig_raw or "<missing>"))

signals_payload = None
if sig_raw:
    try:
        signals_payload = json.loads(sig_raw)
        record("latest_signals JSON valide", True, f"keys = {sorted(signals_payload.keys())}")
    except json.JSONDecodeError as e:
        record("latest_signals JSON valide", False, str(e))

signals = (signals_payload or {}).get("signals", [])
if signals_payload:
    record("signals = list non-vide",
           isinstance(signals, list) and len(signals) > 0,
           f"n_signals = {len(signals) if isinstance(signals, list) else 'not-a-list'}")

for s in signals[:6]:  # cap à 6 pour ne pas spammer
    tenor = s.get("tenor", "?")
    st = s.get("signal_type")
    sigma_mid = s.get("sigma_mid")
    sigma_fair = s.get("sigma_fair")
    dte = s.get("dte")
    finite = (
        isinstance(sigma_mid, (int, float)) and not math.isnan(sigma_mid)
        and isinstance(sigma_fair, (int, float)) and not math.isnan(sigma_fair)
    )
    record(f"  signal[{tenor}] signal_type ∈ {{CHEAP, FAIR, EXPENSIVE}}",
           st in EXPECTED_SIGNAL_TYPES, f"got {st!r}")
    record(f"  signal[{tenor}] sigma_mid/sigma_fair finis",
           finite, f"sigma_mid={sigma_mid}, sigma_fair={sigma_fair}")
    record(f"  signal[{tenor}] dte > 0",
           isinstance(dte, int) and dte > 0, f"dte={dte}")

== 8. latest_signals ==
  [OK  ] latest_signals:EURUSD présent  | {"symbol": "EURUSD", "timestamp": "2026-04-29T09:21:24.277633Z", "signals": []}
  [OK  ] latest_signals JSON valide  | keys = ['signals', 'symbol', 'timestamp']
  [FAIL] signals = list non-vide  | n_signals = 0


## 9. Cohérence : un signal par tenor avec atm

Pour chaque tenor public dans `surface` qui a un `atm.iv` valide, il doit y avoir exactement un signal correspondant. C'est le contrat structurel entre `_derive_signals` et le shape de la surface.

In [32]:
print("== 9. cohérence tenors ↔ signals ==")

tenors_with_atm = {
    t for t, p in surface.items()
    if not t.startswith("_") and isinstance(p, dict)
    and isinstance(p.get("atm"), dict)
    and isinstance(p["atm"].get("iv"), (int, float))
}
tenors_in_signals = {s.get("tenor") for s in signals if isinstance(s, dict)}

record("tenors(signals) ⊆ tenors(surface_with_atm)",
       tenors_in_signals <= tenors_with_atm,
       f"signals={tenors_in_signals}, surface_atm={tenors_with_atm}")
record("tenors(surface_with_atm) ⊆ tenors(signals)",
       tenors_with_atm <= tenors_in_signals,
       f"surface_atm={tenors_with_atm}, signals={tenors_in_signals}")

== 9. cohérence tenors ↔ signals ==
  [OK  ] tenors(signals) ⊆ tenors(surface_with_atm)  | signals=set(), surface_atm={'5M', '3M', '6M', '2M', '4M', '1M'}
  [FAIL] tenors(surface_with_atm) ⊆ tenors(signals)  | surface_atm={'5M', '3M', '6M', '2M', '4M', '1M'}, signals=set()


## Récap final

In [33]:
n_ok = sum(1 for _, ok, _ in results if ok)
n_fail = sum(1 for _, ok, _ in results if not ok)

print(f"\n{'LABEL':<60} STATUS  DETAIL")
print("-" * 120)
for label, ok, detail in results:
    sym = "OK" if ok else "FAIL"
    print(f"{label:<60} {sym:<6}  {detail}")
print("-" * 120)
print(f"\n{n_ok} OK / {n_fail} FAIL  ({len(results)} total)")

if n_fail == 0:
    print("\nOK — Redis outputs vol-engine sains. Pass au notebook 03 (pub/sub channel vol_update).")


LABEL                                                        STATUS  DETAIL
------------------------------------------------------------------------------------------------------------------------
heartbeat:vol_engine présent                                 OK      2026-04-29T09:21:24.280240Z
heartbeat ISO-8601 parsable                                  OK      parsed = 2026-04-29T09:21:24.280240Z
heartbeat age < 200.0s                                       OK      age = 0.4s
heartbeat TTL > 0                                            OK      TTL = 30s
latest_vol_surface:EURUSD présent                            OK      {"symbol": "EURUSD", "timestamp": "2026-04-29T09:21:24.277633Z", "surface": {"1M...
latest_vol_surface JSON valide                               OK      keys = ['surface', 'symbol', 'timestamp']
top-level keys = {symbol, timestamp, surface}                OK      actual = ['surface', 'symbol', 'timestamp']
top-level symbol = EURUSD                                    OK

## Troubleshooting cheat sheet

| Symptôme | Cause probable | Fix |
|---|---|---|
| heartbeat absent ou stale > 200s | engine cycle skip permanent | `docker logs fxvol-vol-engine --tail 100` ; suspects : `latest_spot` absent (market-data down), IB chain vide (TrustedIPs KO ou hors marché) |
| `latest_vol_surface` absent | aucun cycle complet réussi | si IB chain vide → check Trusted IPs, weekend, ou re-test en heures FX |
| tenors public = 0 | chain IB ne renvoie aucune observation utilisable | `docker exec fxvol-vol-engine python -c "from engines.vol.chain_fetcher import ..." ` (debug manuel) |
| `_svi[tenor].butterfly_g_min < 0` répété | dataset chain pourri (3 strikes alignés, IV bruitées) | normal en dev paper sur petits comptes ; en prod investiguer |
| `_har`/`_garch`/`_rv` absents | OHLC fetch fail (perms IB, weekend) | `docker logs fxvol-vol-engine | grep -i ohlc` ; si "failed" répété → permissions IB historical |
| `_fair_q` vide alors que `_har` présent | bug VRP regime detection | regarder `core.vol.vrp:detect_regime` |
| signals vide alors que tenors présents | `iv_mid` non-numeric ou `_fair_q` ET `_garch` vides | fallback path cassé ; check engine logs |
| signal_type tous = FAIR | THRESHOLD_VOL_PTS trop large vs spread mid/fair réel | publier `config:changed` avec un threshold plus serré (notebook 05) |